## Plotting Pressure against baryon density for symmetric nuclear matter including He-6 at T = 10 MeV

The pressure including non-interacting He-6 nuclei is calculated at 3 different levels: first for an ideal proton, neutron and alpha gas, then for just nucleons and nucleon-nucleon interactions, and finally for the full virial expansion. This is done for T = 10 MeV and Y_p = 0.5, however the general code has been used so that Y_p can easily be modified.

In [1]:
# Imports

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import brentq


In [ ]:
# Define constants

T = 10.0 # Temp [MeV]
m_p = 938.272 # mass of proton [MeV/c^2]
lamda = 197.326 * (2 * np.pi/(m_p * T))**(1/2) # nucleon thermal wavelength [fm^-1] assuming m_p=m_n
lamda_alpha = lamda / 2 # alpha particle thermal wavelength [fm^-1]
lamda_beta = lamda/np.sqrt(6) # beta particle thermal wavelength [fm^-1]
lamda_gamma = lamda / (2 * np.sqrt(2)) # gamma particle thermal wavelength [fm^-1]
E_alpha = 28.3 # alpha particle binding energy [MeV]
E_beta = 29.27 # beta particle binding energy [MeV]
E_gamma = 31.41 # gamma particle binding energy [MeV]

Y_p = 0.5 # proton fraction

# virial coefficients for T = 10 MeV
b_n = 0.311
b_pn = 1.91
b_nuc = 2.22
b_alpha_n = 2.95
b_alpha = 20.4

In [ ]:

# Calculate fugacities

#-----------------------
# free n, p, alpha, beta gas
#-----------------------

z_n_1 = np.linspace(1e-8, 0.23, 100) # range of neutron fugacities starting at small positive value

# ideal quadratic coefficients

a_1 = (((16*Y_p - 8)/(1 - Y_p)) * z_n_1**2 * np.exp(E_alpha/T)) + (6*np.sqrt(6)*((3*Y_p - 1)/(1 - Y_p)) * z_n_1**4 * np.exp(E_beta/T)) + ((16*np.sqrt(2)) * ((4*Y_p - 1)/(1 - Y_p)) * z_n_1**6 * np.exp(E_gamma/T))
b_1 = -1.0
c_1 = Y_p * z_n_1/(1 - Y_p)


z_p_1 = (2 * c_1) / (-b_1 + np.sqrt(b_1**2 - 4*a_1*c_1)) # proton fugacity calculated as roots of quadratic equation


z_alpha_1 = z_n_1**2 * z_p_1**2 * np.exp(E_alpha/T) # alpha particle fugacity

z_beta_1 = z_n_1**2 * z_alpha_1 * np.exp((E_beta - E_alpha)/T) # beta particle fugacity

z_gamma_1 = z_n_1**4 * z_alpha_1 * np.exp((E_gamma - E_alpha)/T) # gamma particle fugacity

# free gas validity

valid_1 = ((z_n_1 < 0.23) & (z_p_1 < 0.6) & (z_alpha_1 < 0.17))

# RECALCULATE MAX FUCACITY
# z_n_max = (1 / (24 * np.sqrt(6) * np.exp(E_beta/T)))**(1/5)

print("Maximum neutron fugacity:", z_n_max)

#---------------------------------------
# b_n, b_pn, b_nuc and zero alphas and betas
#---------------------------------------

z_n_2 = np.linspace(1e-8, 0.51, 100) # range of neutron fugacities starting at small positive value


a_2 = -2 * b_n
b_2 = -1 - z_n_2 * b_pn
c_2 = z_n_2 + 2 * z_n_2**2 * b_n

discriminant_2 = b_2**2 - 4*a_2*c_2

z_p_2 = (2*c_2) / (-b_2 + np.sqrt(discriminant_2))

z_alpha_2 = np.zeros_like(z_n_2)
z_beta_2 = np.zeros_like(z_n_2)

valid_2 = (z_n_2 < 0.51) & (z_p_2 < 0.6)


#-------------------------------
# with all interactions
#-------------------------------

z_n_4 = np.linspace(1e-8, 0.2, 100) # range of neutron fugacities starting at small positive value

z_p_4 = []

for zn in z_n_4:
    a = ((32 * Y_p - 16)/(1 - Y_p)) * b_alpha * zn**4 * np.exp(2 * E_alpha/T)
    b = (32 * Y_p - 16)/(1 - Y_p) * b_alpha_n * zn**2 * np.exp(E_alpha/T) - 8 * b_alpha_n * zn**2 * np.exp(E_alpha/T)
    c = ((8 * Y_p)/(1 - Y_p)) * b_alpha_n * zn**3 * np.exp(E_alpha/T) + ((16 * Y_p - 8)/(1 - Y_p)) * (zn**2 * np.exp(E_alpha/T) + 2 * zn**3 * np.exp(E_alpha/T) * b_alpha_n) + (6*np.sqrt(6)*((3*Y_p - 1)/(1 - Y_p)) * zn**4 * np.exp(E_beta/T)) - 2 * b_n
    d = ((2 * Y_p)/(1 - Y_p)) * zn * b_pn - 1 - 2 * (zn * b_pn)
    e = (Y_p/(1 - Y_p)) * (zn + 2 * zn**2 * b_n)

    def g(z_p):
        return a * z_p**4 + b * z_p**3 + c * z_p**2 + d * z_p + e

    z_p = brentq(g, 0.0, 0.6)
    z_p_4.append(z_p)
z_p_4 = np.array(z_p_4)


z_alpha_4 = z_n_4**2 * z_p_4**2 * np.exp(E_alpha/T) # alpha particle fugacity

z_beta_4 = z_n_4**2 * z_alpha_4 * np.exp((E_beta - E_alpha)/T) # beta particle fugacity

# validity for all interactions

valid_4 = ((z_n_4 < 0.21) & (z_p_4 < 0.6) & (z_alpha_4 < 0.04))